# 05 — GSEA / pathway over-representation (real rules)

**Section:** Evaluation layer
**Depends on:** the four rule pools, `03_promethee_ranking.ipynb` (ranked table), `feature_map.json`
**Methodology:** §4.6.3, GSEApy over-representation of the genes in each pool's rules against KEGG

## What is tested, and what is not

- **Gene-level test (mutation-side conditions only).** Genes come from conditions on `<GENE>_mut` features (real HGNC symbols). The **background is the 534 COSMIC driver genes** the mutation features were drawn from (not the whole genome), because rules can only ever select from those 534.
- **A "Low" mutation condition means "not mutated".** Such genes are not evidence that the gene's biology matters, so the **primary query list uses only genes required to be mutated (`High`)**. A list with all mutation genes is a secondary analysis.
- **Pathway-side conditions are not gene lists.** The 38 pathway-activity features are KEGG pathways already (ssGSEA scores), so they are reported as direct pathway usage in §3, with no enrichment test.

## Network note
The Enrichr / KEGG service is **not reachable from the sandbox where the other notebooks were run**. Cell 4 queries Enrichr and, if it cannot connect, prints a message and still saves the query lists so the identical analysis can be run where there is network access (or by pasting the lists into the Enrichr web page). No enrichment result is fabricated or substituted.

In [1]:
import os, sys, json
from pathlib import Path
if Path.cwd().name == "notebooks":
    os.chdir("..")            # run from the repo root so relative paths match the pipeline
sys.path.insert(0, ".")
import numpy as np, pandas as pd

import gseapy as gp
from src.evaluation import stage5_utils as U
pools, rule_lookup, feature_map = U.load_pools()
ranked = pd.read_csv(U.REPORT_DIR / "promethee_ranked_rules.csv")
mut_genes_all = sorted({v["gene_symbol"] for k, v in feature_map.items() if k.endswith("_mut")})
print("background (driver genes available to rules):", len(mut_genes_all))

background (driver genes available to rules): 534


## 1. Genes appearing in each pool's rules

In [2]:
rows = []
for name, pool in pools.items():
    for r in pool["rules"]:
        for c in r["conditions"]:
            if c["feature"].endswith("_mut"):
                rows.append({"pool": name, "rule": r["id"], "gene": feature_map[c["feature"]]["gene_symbol"], "partition": c["partition"]})
genes_long = pd.DataFrame(rows)
genes_long["condition"] = genes_long.pool.str.split("_").str[1]
genes_long["extractor"] = genes_long.pool.str.split("_").str[0]
print("mutation-side conditions:", len(genes_long), "| of which 'High' (mutated):", int((genes_long.partition == "High").sum()))
summary = genes_long.groupby("pool").agg(conditions=("gene", "size"), genes=("gene", "nunique"),
            mutated_High=("partition", lambda s: int((s == "High").sum())))
summary

mutation-side conditions: 25 | of which 'High' (mutated): 11


,conditions,genes,mutated_High
pool,,,
ga_baseline,9,8,6
ga_sam,10,10,1
nsga_baseline,3,2,3
nsga_sam,3,3,1


In [3]:
def lists(df):
    return {
        "mutated (High) genes": sorted(df.loc[df.partition == "High", "gene"].unique()),
        "all mutation genes":   sorted(df["gene"].unique()),
    }
query = {}
for name in pools: query[name] = lists(genes_long[genes_long.pool == name])
for cond in ("baseline", "sam"): query[f"{cond} (GA+NSGA-II)"] = lists(genes_long[genes_long.condition == cond])
ql = pd.DataFrame([{"query": k, "list_type": t, "n_genes": len(g), "genes": ", ".join(g)} for k, d in query.items() for t, g in d.items()])
ql.to_csv(U.REPORT_DIR / "gsea_query_gene_lists.csv", index=False)
ql

,query,list_type,n_genes,genes
0,ga_baseline,mutated (High) genes,5,"ATR, MYC, PAX3, POLE, SPOP"
1,ga_baseline,all mutation genes,8,"ATR, FOXP1, HERPUD1, MYC, PAX3, POLE, SPOP, YWHAE"
2,ga_sam,mutated (High) genes,1,QKI
3,ga_sam,all mutation genes,10,"CBLB, CHEK2, CNTRL, FLT4, GNA11, IL6ST, NCOA1,..."
4,nsga_baseline,mutated (High) genes,2,"GOLGA5, PPP6C"
5,nsga_baseline,all mutation genes,2,"GOLGA5, PPP6C"
6,nsga_sam,mutated (High) genes,1,DDX3X
7,nsga_sam,all mutation genes,3,"DDX3X, EPS15, MUTYH"
8,baseline (GA+NSGA-II),mutated (High) genes,7,"ATR, GOLGA5, MYC, PAX3, POLE, PPP6C, SPOP"
9,baseline (GA+NSGA-II),all mutation genes,10,"ATR, FOXP1, GOLGA5, HERPUD1, MYC, PAX3, POLE, ..."


With 0–7 genes per list, any over-representation test has very little power: with a 534-gene background, a gene set must contain nearly all of a short list to reach significance. Results should be read as exploratory.

## 2. Over-representation against KEGG (needs network access to Enrichr)

In [4]:
LIBRARY = "KEGG_2021_Human"
results, failed = [], None
for qname, d in query.items():
    for ltype, glist in d.items():
        if len(glist) < 2:
            continue
        try:
            res = gp.enrichr(gene_list=glist, gene_sets=LIBRARY, background=mut_genes_all, outdir=None).results
        except Exception as e:                         # no network / service unavailable
            failed = f"{type(e).__name__}: {str(e)[:200]}"
            break
        res.insert(0, "list_type", ltype); res.insert(0, "query", qname)
        results.append(res)
    if failed: break
if failed:
    print("Enrichr not reachable from this environment ->", failed)
    print("The query lists are saved in", U.REPORT_DIR / "gsea_query_gene_lists.csv", "- re-run this notebook where Enrichr is reachable.")
    enr = pd.DataFrame()
else:
    enr = pd.concat(results, ignore_index=True)
    enr.to_csv(U.REPORT_DIR / "gsea_enrichment_results.csv", index=False)
    print("columns returned:", list(enr.columns))
    cols = [c for c in ["query", "list_type", "Term", "Overlap", "P-value", "Adjusted P-value", "Genes"] if c in enr.columns]
    print(enr.sort_values("Adjusted P-value").head(15)[cols].to_string(index=False))
    print("\nterms with adjusted p < 0.05:", int((enr["Adjusted P-value"] < 0.05).sum()), "of", len(enr))

columns returned: ['query', 'list_type', 'Gene_set', 'Term', 'P-value', 'Adjusted P-value', 'Old P-value', 'Old adjusted P-value', 'Odds Ratio', 'Combined Score', 'Genes']
           query            list_type                                                     Term   P-value Adjusted P-value         Genes
sam (GA+NSGA-II) mutated (High) genes                    RIG-I-like receptor signaling pathway  0.018656         0.055969         DDX3X
        nsga_sam   all mutation genes                                     Base excision repair  0.016791         0.069698         MUTYH
        nsga_sam   all mutation genes                    RIG-I-like receptor signaling pathway  0.027879         0.069698         DDX3X
sam (GA+NSGA-II) mutated (High) genes                                              Hepatitis B  0.158146         0.158146         DDX3X
sam (GA+NSGA-II) mutated (High) genes                                     Viral carcinogenesis  0.154703         0.158146         DDX3X
        nsga

## 3. Pathway-feature usage (no external database needed)

Which of the 38 KEGG pathway-activity features the rules condition on, and in which direction.

In [5]:
rows = []
for name, pool in pools.items():
    for r in pool["rules"]:
        for c in r["conditions"]:
            if c["feature"].endswith("_activity"):
                rows.append({"pool": name, "rule": r["id"], "pathway": feature_map[c["feature"]]["gene_symbol"], "partition": c["partition"]})
pw = pd.DataFrame(rows)
usage = (pw.groupby(["pathway", "partition"]).agg(rules=("rule", "nunique"), pools=("pool", lambda s: ", ".join(sorted(set(s)))))
           .reset_index().sort_values("rules", ascending=False))
usage.to_csv(U.REPORT_DIR / "pathway_feature_usage.csv", index=False)
usage

,pathway,partition,rules,pools
1,B cell receptor signaling pathway,High,8,"ga_baseline, ga_sam"
0,Axon guidance,Low,4,"ga_baseline, nsga_baseline"
4,Estrogen signaling pathway,Low,3,ga_sam
6,Natural killer cell mediated cytotoxicity,High,3,"ga_baseline, ga_sam"
3,Cellular senescence,High,1,nsga_sam
2,C-type lectin receptor signaling pathway,High,1,ga_sam
5,Gap junction,Low,1,nsga_sam
7,Regulation of lipolysis in adipocytes,Low,1,nsga_sam
8,Th17 cell differentiation,High,1,nsga_sam
9,Thyroid hormone signaling pathway,High,1,ga_baseline


In [6]:
# how much of the pool depends on each pathway, and is the same pathway used under both models?
share = pw.groupby(["pool"]).apply(lambda d: d.groupby("pathway").rule.nunique() / len(pools[d.name]["rules"])).unstack(0).fillna(0).round(2)
share.loc[share.max(axis=1) >= 0.3]

pool,ga_baseline,ga_sam,nsga_baseline,nsga_sam
pathway,,,,
Axon guidance,0.14,0.00,1.0,0.00
B cell receptor signaling pathway,0.71,0.50,0.0,0.00
Cellular senescence,0.00,0.00,0.0,0.33
Estrogen signaling pathway,0.00,0.50,0.0,0.00
Gap junction,0.00,0.00,0.0,0.33
Natural killer cell mediated cytotoxicity,0.14,0.33,0.0,0.00
Regulation of lipolysis in adipocytes,0.00,0.00,0.0,0.33
Th17 cell differentiation,0.00,0.00,0.0,0.33


## Interpretation notes

- Report the pathway usage (§3) as the main biologically readable output: which KEGG pathways the model's sensitive-region rules rely on, and whether SAM and baseline rely on the same ones.
- Report the gene-level enrichment (§2) with its power limitation. A null result there is expected at these list sizes and is not evidence against biological relevance.
- Rules are drug-blind and cover about 20 cell lines at the floor, so a pathway such as B-cell receptor signalling may be marking a lineage (haematological cell lines) rather than a drug-specific mechanism. That is a hypothesis for future work (needs tissue annotation), not a finding.